# Autoresearch

An agent edits one file, measures, keeps or discards — and repeats without you.

*You are the slow part of research, not the thinking.*

> Faithful to [karpathy/autoresearch](https://github.com/karpathy/autoresearch): one loop.

## 0. Setup

Reading and writing are separate permissions — this split *is* the design:

| path | reads | writes |
|---|---|---|
| `program.md` | **yes** — every prompt | no |
| `src/` | **yes** | **yes** — on a win |
| `harness/` | no | no |

In [1]:
import os
import re
from pathlib import Path

from dotenv import find_dotenv, load_dotenv
from openai import OpenAI

from harness.measure import measure   # protected/ — never enters a prompt

load_dotenv(find_dotenv(usecwd=True))
client = OpenAI(
    api_key=os.environ["DEEPINFRA_API_KEY"],
    base_url="https://api.deepinfra.com/v1/openai",
)
MODEL = "meta-llama/Llama-4-Maverick-17B-128E-Instruct-FP8"   # see Weaknesses: 70B fails this task

SOLVE = Path("src/solve.py")   # the only file the agent may rewrite
PROGRAM = Path("program.md").read_text()
MAX_EXPERIMENTS, PATIENCE = 15, 8

## 1. The boundary — load both and look

The agent reads `program.md` and `src/`. It never sees `harness/`, so the verifier can't be
argued with.

In [ ]:
EDITABLE  = sorted(f for f in Path("src").iterdir() if f.is_file())
PROTECTED = sorted(f for f in Path("harness").iterdir() if f.is_file())

for label, paths, seen in [("EDITABLE  src/", EDITABLE, "the agent rewrites these"),
                           ("PROTECTED harness/", PROTECTED, "never enters a prompt")]:
    print(f"{'='*70}\n{label}  — {seen}\n{'='*70}")
    for f in paths:
        print(f"--- {f} ---")
        print(f.read_text().rstrip(), "\n")

## 2. The prompt

Everything the agent gets: `program.md`, the editable file, the last measurement. Nothing else.

In [2]:
def propose(src, seconds, note):
    """The only thing the agent ever sees: program.md + solve.py + the last measurement."""
    msg = (f"{PROGRAM}\n\n"
           f"src/solve.py:\n```python\n{src}\n```\n"
           f"Current: {seconds*1000:.3f} ms. Last result: {note}\n"
           f"Propose ONE change. Reply with only the new `def solve(n):` in a ```python block.")
    out = client.chat.completions.create(
        model=MODEL, max_tokens=500, messages=[{"role": "user", "content": msg}],
    ).choices[0].message.content
    m = re.search(r"```python\n(.*?)```", out, re.S)
    return m.group(1).strip() if m else None

## 3. The loop

Measure, then write only on a win — so `src/solve.py` only ever holds the best.

In [ ]:
def autoresearch():
    best = SOLVE.read_text()
    best_s, note = measure(best)
    stale = 0

    for i in range(1, MAX_EXPERIMENTS + 1):
        cand = propose(best, best_s, note)
        secs, note = measure(cand) if cand else (None, "no code block")
        kept = secs is not None and secs < best_s

        if kept:
            SOLVE.write_text(cand)          # commit: the file only ever holds the best
            best, best_s, stale = cand, secs, 0
        else:
            stale += 1                      # discard: nothing was written, nothing to undo

        print(f"  {i:>2} {('%.3f ms' % (secs*1000)) if secs else '—':>12}  "
              f"{'KEEP ' if kept else 'discard'}  best={best_s*1000:.3f} ms  {note[:44]}")

        if stale >= PATIENCE:
            print(f"  stop: no improvement in {PATIENCE} experiments")
            break
    return best, best_s

## 4. Run

`git diff src/` afterwards to see what it did.

In [ ]:
base = SOLVE.read_text()
base_s, _ = measure(base)
print(f"baseline {base_s*1000:.3f} ms\n")

best, best_s = autoresearch()
print(f"\nbaseline {base_s*1000:.3f} ms -> best {best_s*1000:.3f} ms  ({base_s/best_s:.0f}x)")
print(best)

## Weaknesses

| Weakness | What happens | Fix |
|---|---|---|
| **The model is the ceiling** | `Llama-3.1-70B` never solves this — same off-by-one every attempt (`WRONG on n=10: got 33, want 23`). Maverick and DeepSeek-V3.2 land it first try | Use a model that can do the task |
| **The boundary is convention** | `exec` runs model code in-process with write access to `harness/` | Subprocess, read-only mount, timeout |
| **The metric is the objective** | It optimises `measure`'s return, not your intent. Correctness is pass/fail — no partial credit | Verify the verifier first |
| **Timing is noisy** | Baseline spreads `86.6–88.9 ms` (3%), so a 3% "gain" is noise and gets committed | Min of N, or a threshold |
| **No rollback** | The first win overwrites the baseline, and this dir is untracked | Commit before running |
| **No memory** | Each proposal sees only the best and the last note, so it retries dead ideas | Pass the log |
| **One loop, by design** | The strategy never changes; nothing notices a stall | none — a second loop is a different mechanism |

## Notes

Min of 5, on this machine:

| | ms |
|---|---|
| naive baseline | `87.57` |
| what the loop found | `0.0004` (~233,000×) |

It landed the closed form with the correct off-by-one (`(n-1)//m`) — what `Llama-3.1-70B`
failed at three times.

Unverified: `propose` assumes one ```python block.